<a href="https://colab.research.google.com/github/ubsuny/PHY386/blob/Homework2026/2026/FinalProject/Final_03_MethaneCH4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Final Project — Topic 3: Atmospheric Methane (CH₄) Forecast (42 pts)
## The regression / time-series stretch

## Learning Outcomes
- Decompose a real climate time series into trend, seasonal, and residual components.
- Fit physically motivated models (linear, quadratic, Fourier-series seasonal) with `scipy.optimize.curve_fit` and report parameter uncertainties.
- Extrapolate a regression model with a **quantified confidence band** — not just a point forecast.
- Recognize when a simple model fails (here: the 1999–2006 methane plateau) and address it with a piecewise or change-point model.
- Compare a purely statistical machine-learning approach (sklearn `MLPRegressor` on lagged features) with the physically motivated fit.

## GitHub Workflow
Fork → `yourname-final` → PR into `Homework2026`. Reviewer `@laserlab`, milestone `Final-2026`.

## Why this topic is the "stretch"
HW5 and HW6 covered **classification** only. Here you'll work on **regression on a time series**, which requires a different metric set (root-mean-square error, prediction intervals — not confusion matrices) and different diagnostics (residual plots, autocorrelation). That's the point: you'll pick up new methodology on top of the HW5/HW6 toolkit.

## Background: Atmospheric Methane

Methane (CH₄) is the second-most-important long-lived greenhouse gas after CO₂. Per molecule, it's roughly **80× more potent than CO₂ as a climate forcer over a 20-year horizon**, though it has a much shorter atmospheric lifetime (~10 yr vs. centuries for CO₂).

The NOAA Global Monitoring Laboratory (GML) publishes a **globally-averaged marine-surface** methane record going back to **July 1983**. The time series has three physically distinct features you will model:

1. **Long-term trend** — driven by the global source minus sink balance. Not a clean linear rise:
   - ~1983–1999: rapid rise (~12 ppb/yr).
   - **~1999–2006: plateau** — sources and sinks roughly balanced. The physics is still debated: shifts in wetland emissions, tropical biosphere response, perhaps changes in OH-radical concentration.
   - **~2007–present: renewed rise**, accelerating sharply around **2020**.
2. **Seasonal cycle** — CH₄ has a small (~10 ppb peak-to-peak) annual cycle driven by photochemical oxidation by OH in summer.
3. **Inter-annual variability** — the residual after removing trend and seasonal, visibly correlated with El Niño.

A student who naively fits a single polynomial to this series and extrapolates will get 2030 catastrophically wrong. Diagnosing *why* is the core learning outcome.

In [ ]:
# Required packages (uncomment for Colab or fresh environment):
# %pip install pandas scipy statsmodels scikit-learn --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

plt.rcParams.update({
    'font.size': 11, 'axes.labelsize': 11, 'axes.titlesize': 11,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 11,
})

## Worked Example: Load the NOAA GML Methane Record

In [ ]:
URL = "https://gml.noaa.gov/webdata/ccgg/trends/ch4/ch4_mm_gl.txt"
df = pd.read_csv(
    URL,
    comment="#",
    sep=r"\s+",
    names=["year", "month", "decimal",
           "average", "average_unc",
           "trend", "trend_unc"],
)
# NOAA encodes missing uncertainty as -9.9; drop any such rows defensively.
df = df[df["average_unc"] > 0].reset_index(drop=True)
print(f"{len(df)} monthly points from {df['decimal'].min():.2f} to {df['decimal'].max():.2f}")
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(df["decimal"], df["average"], lw=0.8, label="monthly mean")
ax.plot(df["decimal"], df["trend"], lw=1.5, label="NOAA de-seasoned trend")
ax.set_xlabel("year")
ax.set_ylabel("CH$_4$ (ppb)")
ax.set_title("NOAA GML globally-averaged methane")
ax.legend()
ax.tick_params(which='both', direction='in', top=True, right=True)
plt.tight_layout()
plt.show()

## Part 1 — Time-Series Decomposition (8 pts)

### Task 1.1 — Seasonal-trend decomposition (4 pts)
Use `statsmodels.tsa.seasonal_decompose(df['average'], period=12, model='additive')`. Plot the four panels (observed, trend, seasonal, residual) stacked vertically.

### Task 1.2 — Seasonal amplitude (4 pts)
What is the peak-to-peak amplitude of the seasonal component? In which months does CH₄ peak and trough? Briefly (2–3 sentences) relate this to the OH-oxidation seasonal cycle.

In [ ]:
# Part 1: your code here
# from statsmodels.tsa.seasonal import seasonal_decompose
# decomp = seasonal_decompose(df['average'].values, period=12, model='additive')

## Part 2 — Parametric Fit (14 pts)

### Task 2.1 — Linear trend + single-harmonic seasonal (4 pts)
Fit the monthly mean to:
$$\mathrm{CH_4}(t) = a + b\,t + A\cos(2\pi t) + B\sin(2\pi t)$$
with $t$ in years (use the `decimal` column). Report the fitted parameters ± 1σ. Plot the fit over the data.

### Task 2.2 — Quadratic trend (4 pts)
Same model with $a + b t + c t^2$ instead of $a + b t$. Compare the residuals visually and via the root-mean-square error. Is the quadratic better? Does it capture the 1999–2006 plateau?

### Task 2.3 — Piecewise (change-point) trend (6 pts)
Fit a **three-segment** linear trend with change points at **1999.0** and **2007.0**, keeping the Fourier-series seasonal component. You can do this two ways:
- Hand-rolled: slice the DataFrame into three epochs and fit each separately.
- Joint fit: write a piecewise function and let `curve_fit` estimate the slopes simultaneously.

Report the three slopes (ppb/yr) with uncertainties. Do they match the textbook phases described in the Background section?

In [ ]:
# Part 2: your code here
#
# def model_linear(t, a, b, A, B):
#     return a + b*t + A*np.cos(2*np.pi*t) + B*np.sin(2*np.pi*t)
# popt, pcov = curve_fit(model_linear, df['decimal'], df['average'], sigma=df['average_unc'], absolute_sigma=True)

## Part 3 — Extrapolation with Uncertainty (8 pts)

### Task 3.1 — Extrapolate each model to 2030 (4 pts)
Using the covariance matrix from `curve_fit`, propagate the parameter uncertainty to a **2030 CH₄ forecast** for each of your three models. Report central value ± 1σ and ± 2σ.

### Task 3.2 — Held-out cross-validation (4 pts)
Refit each model on data up through **2019** only. Predict 2020–2024 and compare to the actual values. Which model generalizes best? Plot forecast vs. actual with a shaded confidence band.

In [ ]:
# Part 3: your code here

## Part 4 — Machine Learning Benchmark (8 pts)

### Task 4.1 — Lagged-feature Multi-Layer Perceptron (5 pts)
Build features from **lags** of the series: for each target month, use CH₄ values from 1, 2, 3, 6, and 12 months earlier. Train sklearn's `MLPRegressor` (same Multi-Layer Perceptron you used in HW6, just switched to regression) with two hidden layers of 32 units each. Report root-mean-square error on a held-out test split.

### Task 4.2 — Comparison (3 pts)
Put all four models (linear+seasonal, quadratic+seasonal, piecewise+seasonal, `MLPRegressor`) side-by-side in a summary table:

| Model | Train RMSE (ppb) | Test RMSE (ppb) | 2030 forecast (ppb) | 2030 ±1σ |
|-------|------------------|-----------------|---------------------|----------|
| ...   |                  |                 |                     |          |

Which model do you trust most for the 2030 forecast, and why?

In [ ]:
# Part 4: your code here
# from sklearn.neural_network import MLPRegressor
# from sklearn.preprocessing import StandardScaler

## Part 5 — Physics Discussion (4 pts)

Write ~8 sentences addressing:
- Why does the naive linear (or quadratic) extrapolation fail? Tie your residual plot to the textbook plateau-then-re-acceleration story.
- The Multi-Layer Perceptron has no physical model baked in — how would it do on genuinely novel behavior, e.g. a sudden drop caused by policy action?
- If you include CO₂ as a second signal (bonus: load `statsmodels.datasets.co2`), which gas is easier to forecast and why?

### Stretch (optional, bonus 4 pts)
Compare CH₄ and CO₂ on the same plot. Align them on a common time axis and comment on the relative forecast uncertainty in 2030.

## Submission
**Good commit** ✅: `Add piecewise trend fit with three change points`

**Bad commit** ❌: `done`, `update`

### Checklist
- [ ] Notebook in `2026/FinalProject/<yourname>/`
- [ ] Runs top-to-bottom on a fresh kernel
- [ ] Functions have type annotations and NumPy docstrings
- [ ] All plots have labels with units; residual plots shown for each fit
- [ ] Tasks sum to ~42 points
- [ ] PR into `Homework2026`, reviewer `@laserlab`, milestone `Final-2026`
- [ ] AI usage disclosed in the PR

## Resources
- [NOAA GML methane trends page](https://gml.noaa.gov/ccgg/trends_ch4/)
- [`statsmodels.tsa.seasonal_decompose`](https://www.statsmodels.org/stable/generated/statsmodels.tsa.seasonal.seasonal_decompose.html)
- [`scipy.optimize.curve_fit`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html)
- [`sklearn.neural_network.MLPRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html)
- Dlugokencky et al. (1994), *Atmospheric methane at Mauna Loa and Barrow observatories*, JGR 99.